# Vertex AI AutoML Tabular Evaluation & Experiment Tracking

This tutorial demonstrates how to inspect evaluation metrics, extract feature attributions, execute Design of Experiments (DoE) campaigns, and track/compare runs in Vertex AI Experiments using `tabflows`.

### Objectives
1. Discover trained models and fetch primary classification evaluation metrics (log loss, ROC AUC, precision, recall).
2. Extract global feature importances / attributions.
3. Execute Design of Experiments (DoE) campaigns using `run_doe_campaign` to track parameter variants.
4. Retrieve and compare experiment runs side-by-side using `list_experiment_runs`.

In [1]:
from dotenv import load_dotenv
from google.cloud import aiplatform

from tabflows import (
    TabularPipelineConfig,
    get_model_evaluation_metrics,
    get_model_feature_attributions,
    list_experiment_runs,
    list_models,
    run_doe_campaign,
)

# Load environment variables from local .env file
load_dotenv()
print("Environment and libraries loaded successfully.")

Environment and libraries loaded successfully.


In [2]:
# TabularPipelineConfig automatically loads GCP_PROJECT, GCP_LOCATION, GCP_BUCKET_URI
config = TabularPipelineConfig()

print(f"Project ID: {config.project_id}")
print(f"Location: {config.location}")
print(f"Experiment Name: {config.experiment_name}")

Project ID: hybrid-vertex
Location: us-central1
Experiment Name: automl-tabular-classification-experiments


In [3]:
# Discover recent trained models in Vertex AI Model Registry
try:
    models = list_models(config=config, limit=5)
    tabular_models = [
        m
        for m in models
        if "tabular" in m.display_name.lower() or "automl" in m.display_name.lower()
    ]
    target_models = tabular_models if tabular_models else models

    if target_models:
        model = target_models[0]
        print(f"Selected Trained Model: {model.display_name} ({model.resource_name})")
    else:
        print("No trained models found in Vertex AI Model Registry.")
except Exception as e:
    print(f"Could not list models: {e}")

Selected Trained Model: tabular-workflow-model-7e373ee4-6f03-48f7-b23d-a3b6e30b6dbf (projects/934903580331/locations/us-central1/models/612894719357222912)


## 1. Model Evaluation Metrics

Inspect classification evaluation metrics generated during model training in Vertex AI Model Registry.

In [4]:
if "model" in locals() and isinstance(model, aiplatform.Model):
    print(f"Fetching evaluation metrics for model '{model.resource_name}'...")
    metrics = get_model_evaluation_metrics(model=model, config=config)
    print("\n--- Model Evaluation Summary ---")
    for key, val in metrics.items():
        if not isinstance(val, (dict, list)):
            print(f"  {key}: {val}")

    if "logLoss" in metrics:
        print(f"\nLog Loss: {metrics.get('logLoss')}")
    if "auPrc" in metrics:
        print(f"PR AUC: {metrics.get('auPrc')}")
    if "auRoc" in metrics:
        print(f"ROC AUC: {metrics.get('auRoc')}")
else:
    print("Notice: Load a valid 'model' object in Cell 2 to inspect evaluation metrics.")

Fetching evaluation metrics for model 'projects/934903580331/locations/us-central1/models/612894719357222912'...

--- Model Evaluation Summary ---
  confusionMatrix: <proto.marshal.collections.maps.MapComposite object at 0x7f3dc16c15d0>
  auPrc: 0.9776611
  confidenceMetrics: [<proto.marshal.collections.maps.MapComposite object at 0x7f3dc16c2250>, <proto.marshal.collections.maps.MapComposite object at 0x7f3dc16c1ad0>, <proto.marshal.collections.maps.MapComposite object at 0x7f3dc16c17d0>, <proto.marshal.collections.maps.MapComposite object at 0x7f3dc16c0dd0>, <proto.marshal.collections.maps.MapComposite object at 0x7f3dc16c14d0>, <proto.marshal.collections.maps.MapComposite object at 0x7f3dc16c1650>, <proto.marshal.collections.maps.MapComposite object at 0x7f3dc16c2e10>, <proto.marshal.collections.maps.MapComposite object at 0x7f3dc16c16d0>, <proto.marshal.collections.maps.MapComposite object at 0x7f3dc16c08d0>, <proto.marshal.collections.maps.MapComposite object at 0x7f3dc16c0810>, <p

## 2. Feature Importance / Attributions

Extract global feature attributions to evaluate feature importance across trained models.

In [5]:
if "model" in locals() and isinstance(model, aiplatform.Model):
    print(f"Fetching global feature attributions for model '{model.resource_name}'...")
    attributions = get_model_feature_attributions(model=model, config=config)
    if attributions:
        print("\n--- Global Feature Importance ---")
        sorted_attributions = sorted(attributions.items(), key=lambda x: x[1], reverse=True)
        for feature, score in sorted_attributions:
            print(f"  {feature:<20}: {score:.4f}")
    else:
        print("No feature attributions available for this model evaluation.")
else:
    print("Notice: Load a valid 'model' object in Cell 2 to inspect feature attributions.")

Fetching global feature attributions for model 'projects/934903580331/locations/us-central1/models/612894719357222912'...
No feature attributions available for this model evaluation.


## 3. Vertex AI Experiment Tracking & Design of Experiments (DoE)

Vertex AI Experiments allows tracking hyperparameter variations, pipeline runs, and evaluation metrics across model iterations.
Use `run_doe_campaign` to systematically log multiple Design of Experiments (DoE) variants, and `list_experiment_runs` to perform side-by-side metric comparisons.

In [6]:
# Execute a Design of Experiments (DoE) campaign tracking hyperparameter variants
doe_variants = [
    {
        "name": "variant_standard_budget",
        "train_budget_milli_node_hours": 1000,
        "optimization_objective": "minimize-log-loss",
    },
    {
        "name": "variant_extended_budget",
        "train_budget_milli_node_hours": 2000,
        "optimization_objective": "maximize-au-prc",
    },
]

print(f"Executing DoE campaign 'bank-marketing-doe' with {len(doe_variants)} variants...")
try:
    doe_results = run_doe_campaign(
        campaign_name="bank-marketing-doe",
        variants=doe_variants,
        config=config,
    )
    print(f"DoE campaign executed successfully. Logged {len(doe_results)} variant runs.")
except Exception as e:
    print(f"Notice: DoE campaign execution status: {e}")

Fetching experiment runs for 'automl-tabular-classification-experiments'...
Notice: Vertex AI Experiment 'automl-tabular-classification-experiments' not yet initialized: 404 Resource not found.; GetContext is unable to find context resource with name: projects/934903580331/locations/us-central1/metadataStores/default/contexts/automl-tabular-classification-experiments [detail: "[ORIGINAL ERROR] generic::not_found: com.google.cloud.ai.platform.common.errors.AiPlatformException: code=NOT_FOUND, message=Resource not found.; GetContext is unable to find context resource with name: projects/934903580331/locations/us-central1/metadataStores/default/contexts/automl-tabular-classification-experiments, cause=null [google.rpc.error_details_ext] { code: 5 message: \"Resource not found.; GetContext is unable to find context resource with name: projects/934903580331/locations/us-central1/metadataStores/default/contexts/automl-tabular-classification-experiments\" }"
]


In [ ]:
# Retrieve and perform side-by-side comparison of experiment runs
print(f"Fetching experiment runs for '{config.experiment_name}'...")
try:
    df = list_experiment_runs(config=config)
    if df is not None and not df.empty:
        print(f"Found {len(df)} experiment run(s):")
        display_cols = [
            c for c in df.columns if any(k in c for k in ["name", "param", "metric", "time"])
        ]
        print(df[display_cols if display_cols else df.columns].head())
    else:
        print(f"No experiment runs logged yet under '{config.experiment_name}'.")
except Exception as e:
    print(f"Notice: Vertex AI Experiment '{config.experiment_name}' status: {e}")